# Mutual Fund Performance Analytics

## Objective

This notebook evaluates the historical performance and risk characteristics of mutual fund schemes using:

- Daily returns
- CAGR
- Sharpe Ratio
- Sortino Ratio
- Alpha and Beta
- Maximum Drawdown
- Composite Fund Scorecard
- Benchmark comparison
- Tracking Error

### Benchmark
Nifty 100

### Risk-free rate
6.5% (RBI repo-rate proxy)

### Analysis Period
2022–2026

In [ ]:
# Imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import linregress

print("Libraries loaded successfully.")

In [ ]:
from pathlib import Path

PROJECT_PATH = Path(
    r"C:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics"
)

DATA_PATH = PROJECT_PATH / "data" / "processed"

print("Data path:")
print(DATA_PATH)

#file verification
print("Files available:")

for file in sorted(DATA_PATH.glob("*.csv")):
    print(" -", file.name)

In [ ]:
#loading first 3 datasets
fund_metadata = pd.read_csv(
    DATA_PATH / "fund_metadata.csv"
)

nav_history = pd.read_csv(
    DATA_PATH / "nav_history.csv"
)

benchmark = pd.read_csv(
    DATA_PATH / "10_benchmark_indices.csv"
)

print("Fund metadata:", fund_metadata.shape)
print("NAV history:", nav_history.shape)
print("Benchmark:", benchmark.shape)

In [ ]:
# Inspecting the columns
print("Fund Metadata")
print(fund_metadata.columns.tolist())

print("\nNAV History")
print(nav_history.columns.tolist())

print("\nBenchmark")
print(benchmark.columns.tolist())

In [ ]:
# cleaning NAV data
nav_history["date"] = pd.to_datetime(
    nav_history["date"],
    errors="coerce"
)

nav_history["nav"] = pd.to_numeric(
    nav_history["nav"],
    errors="coerce"
)

nav_history["scheme_code"] = pd.to_numeric(
    nav_history["scheme_code"],
    errors="coerce"
)
 # removing unusable records
nav_history = nav_history.dropna(
    subset=["date", "nav", "scheme_code"]
)

nav_history = nav_history[
    nav_history["nav"] > 0
]

# Remove duplicate scheme/date observations
nav_history = nav_history.drop_duplicates(
    subset=["date", "scheme_code"]
)

# Sorting
nav_history = nav_history.sort_values(
    ["scheme_code", "date"]
).reset_index(drop=True)

In [ ]:
# Cleaning benchmark data
benchmark["date"] = pd.to_datetime(
    benchmark["date"],
    errors="coerce"
)

benchmark["close_value"] = pd.to_numeric(
    benchmark["close_value"],
    errors="coerce"
)

benchmark = benchmark.dropna(
    subset=["date", "close_value", "index_name"]
)

benchmark = benchmark[
    benchmark["close_value"] > 0
]

benchmark = benchmark.sort_values(
    ["index_name", "date"]
).reset_index(drop=True)

In [ ]:
# how many schemes?
scheme_counts = (
    nav_history.groupby("scheme_code")["date"]
    .agg(
        start_date="min",
        end_date="max",
        observations="count"
    )
    .reset_index()
)

print("Unique schemes in NAV history:")
print(scheme_counts["scheme_code"].nunique())

print("\nScheme coverage:")
display(scheme_counts)

# comparing against metadata
metadata_codes = set(
    fund_metadata["scheme_code"].dropna().astype(int)
)

nav_codes = set(
    nav_history["scheme_code"].dropna().astype(int)
)

print("Schemes in metadata:", len(metadata_codes))
print("Schemes in NAV:", len(nav_codes))

print(
    "Metadata schemes missing from NAV:",
    metadata_codes - nav_codes
)

print(
    "NAV schemes missing from metadata:",
    nav_codes - metadata_codes
)

In [ ]:
# scheme name mapping
scheme_lookup = fund_metadata[
    ["scheme_code", "scheme_name"]
].copy()

scheme_lookup["scheme_code"] = pd.to_numeric(
    scheme_lookup["scheme_code"],
    errors="coerce"
)

scheme_lookup = scheme_lookup.drop_duplicates(
    subset="scheme_code"
)

#merging names into NAV
nav_history = nav_history.merge(
    scheme_lookup,
    on="scheme_code",
    how="left"
)

#verification
nav_history[
    ["scheme_code", "scheme_name"]
].drop_duplicates().head(20)

In [ ]:
# NAV matrix
nav_matrix = nav_history.pivot_table(
    index="date",
    columns="scheme_code",
    values="nav",
    aggfunc="last"
)

nav_matrix = nav_matrix.sort_index()

print("NAV matrix shape:", nav_matrix.shape)

In [ ]:
# coverage of each scheme
coverage = pd.DataFrame({
    "observations": nav_matrix.count(),
    "first_date": nav_matrix.apply(
        lambda x: x.first_valid_index()
    ),
    "last_date": nav_matrix.apply(
        lambda x: x.last_valid_index()
    )
})

coverage = coverage.sort_values(
    "observations",
    ascending=False
)

display(coverage)
print(
    "Schemes with NAV data:",
    coverage["observations"].gt(0).sum()
)

print(
    "Schemes with fewer than 500 observations:",
    coverage["observations"].lt(500).sum()
)

# Calculating Daily returns

In [ ]:
nav_history["daily_return"] = (
    nav_history
    .groupby("scheme_code")["nav"]
    .pct_change()
)

returns = nav_history.dropna(
    subset=["daily_return"]
).copy()

print("Return observations:", len(returns))
print(
    "Schemes:",
    returns["scheme_code"].nunique()
)

display(
    returns.head(10)
)

In [ ]:
# Validating return distribution
plt.figure(figsize=(12, 6))

sns.histplot(
    returns["daily_return"],
    bins=100,
    kde=True
)

plt.title("Distribution of Daily NAV Returns — 34 Schemes")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# scheme lookup
metadata_lookup = fund_metadata[
    ["scheme_code", "scheme_name", "fund_house"]
].copy()

# Standardize scheme code
metadata_lookup["scheme_code"] = pd.to_numeric(
    metadata_lookup["scheme_code"],
    errors="coerce"
)

# Remove invalid codes
metadata_lookup = metadata_lookup.dropna(
    subset=["scheme_code"]
)

# Convert to integer
metadata_lookup["scheme_code"] = (
    metadata_lookup["scheme_code"].astype(int)
)

# Remove duplicate scheme codes
metadata_lookup = metadata_lookup.drop_duplicates(
    subset=["scheme_code"]
)

# Keep only schemes that actually exist in NAV history
valid_scheme_codes = returns["scheme_code"].unique()

metadata_lookup = metadata_lookup[
    metadata_lookup["scheme_code"].isin(valid_scheme_codes)
].copy()

# ------------------------------------------------------------
# Remove any existing metadata columns from returns
# ------------------------------------------------------------

for col in ["scheme_name", "fund_house"]:
    if col in returns.columns:
        returns = returns.drop(columns=col)

# ------------------------------------------------------------
# Merge metadata
# ------------------------------------------------------------

returns = returns.merge(
    metadata_lookup,
    on="scheme_code",
    how="left",
    validate="many_to_one"
)

print("Lookup rows:", len(metadata_lookup))
print("Returns rows:", len(returns))
print("Unique schemes:", returns["scheme_code"].nunique())

print("\nMissing scheme names:")
print(returns["scheme_name"].isna().sum())

print("\nSample:")
display(
    returns[
        ["scheme_code", "scheme_name", "fund_house"]
    ].drop_duplicates().head(10)
)

# CAGR Calculation

In [ ]:
# CAGR function
def calculate_cagr(nav_series, years):
    """
    Calculate CAGR using NAV values approximately
    'years' apart.
    """

    nav_series = nav_series.sort_values("date")

    start_date = nav_series["date"].min()
    end_date = nav_series["date"].max()

    target_start = end_date - pd.DateOffset(years=years)

    start_candidates = nav_series[
        nav_series["date"] >= target_start
    ]

    if start_candidates.empty:
        return np.nan

    start_row = start_candidates.iloc[0]
    end_row = nav_series.iloc[-1]

    actual_years = (
        end_row["date"] - start_row["date"]
    ).days / 365.25

    if actual_years <= 0:
        return np.nan

    return (
        (end_row["nav"] / start_row["nav"])
        ** (1 / actual_years)
    ) - 1

In [ ]:
# CAGR for every scheme
cagr_rows = []

for scheme_code, group in nav_history.groupby(
    "scheme_code"
):

    row = {
        "scheme_code": scheme_code,
        "cagr_1yr": calculate_cagr(group, 1),
        "cagr_3yr": calculate_cagr(group, 3),
        "cagr_5yr": calculate_cagr(group, 5)
    }

    cagr_rows.append(row)

cagr = pd.DataFrame(cagr_rows)

cagr = cagr.merge(
    metadata_lookup,
    on="scheme_code",
    how="left"
)

cagr = cagr[
    [
        "scheme_code",
        "scheme_name",
        "fund_house",
        "cagr_1yr",
        "cagr_3yr",
        "cagr_5yr"
    ]
]

display(cagr.sort_values(
    "cagr_3yr",
    ascending=False
))

# Calculating Sharpe's Ratio

In [ ]:
# Sharpe Ratio
RISK_FREE_ANNUAL = 0.065

RISK_FREE_DAILY = (
    (1 + RISK_FREE_ANNUAL) ** (1 / 252)
) - 1

print("Annual risk-free rate:", RISK_FREE_ANNUAL)
print("Daily risk-free rate:", RISK_FREE_DAILY)

In [ ]:
sharpe_rows = []

for scheme_code, group in returns.groupby(
    "scheme_code"
):

    excess_return = (
        group["daily_return"]
        - RISK_FREE_DAILY
    )

    std_return = group["daily_return"].std()

    if std_return == 0:
        sharpe = np.nan
    else:
        sharpe = (
            excess_return.mean()
            / std_return
        ) * np.sqrt(252)

    sharpe_rows.append({
        "scheme_code": scheme_code,
        "sharpe_ratio": sharpe
    })

sharpe = pd.DataFrame(sharpe_rows)

display(
    sharpe.sort_values(
        "sharpe_ratio",
        ascending=False
    )
)

# Sortino Ratio

In [ ]:
sortino_rows = []

for scheme_code, group in returns.groupby(
    "scheme_code"
):

    daily_returns = group["daily_return"]

    excess_return = (
        daily_returns - RISK_FREE_DAILY
    )

    downside_returns = daily_returns[
        daily_returns < 0
    ]

    downside_std = downside_returns.std()

    if pd.isna(downside_std) or downside_std == 0:
        sortino = np.nan
    else:
        sortino = (
            excess_return.mean()
            / downside_std
        ) * np.sqrt(252)

    sortino_rows.append({
        "scheme_code": scheme_code,
        "sortino_ratio": sortino
    })

sortino = pd.DataFrame(sortino_rows)

display(
    sortino.sort_values(
        "sortino_ratio",
        ascending=False
    )
)

# Maximum Drawdown

In [ ]:
drawdown_rows = []

for scheme_code, group in nav_history.groupby(
    "scheme_code"
):

    group = group.sort_values("date").copy()

    group["running_max"] = (
        group["nav"].cummax()
    )

    group["drawdown"] = (
        group["nav"] /
        group["running_max"]
    ) - 1

    worst_idx = group["drawdown"].idxmin()

    worst_row = group.loc[worst_idx]

    drawdown_rows.append({
        "scheme_code": scheme_code,
        "max_drawdown": worst_row["drawdown"],
        "drawdown_date": worst_row["date"]
    })

drawdown = pd.DataFrame(drawdown_rows)

drawdown = drawdown.merge(
    metadata_lookup,
    on="scheme_code",
    how="left"
)

display(
    drawdown.sort_values(
        "max_drawdown"
    )
)

# Alpha & Beta Calculation

In [41]:
benchmark_name = "Nifty 50"

benchmark_selected = benchmark[
    benchmark["index_name"].str.lower() == benchmark_name.lower()
].copy()

benchmark_selected = benchmark_selected.sort_values("date")

benchmark_selected["benchmark_return"] = (
    benchmark_selected["close"]
    .pct_change()
)

benchmark_selected = benchmark_selected.dropna(
    subset=["benchmark_return"]
)

print(
    benchmark_selected[
        ["date", "close", "benchmark_return"]
    ].head()
)

print(
    "Benchmark observations:",
    len(benchmark_selected)
)
from scipy.stats import linregress

alpha_beta_results = []

for scheme_code, fund_data in returns.groupby("scheme_code"):

    fund_data = fund_data[
        ["date", "daily_return"]
    ].dropna()

    merged = fund_data.merge(
        benchmark_selected[
            ["date", "benchmark_return"]
        ],
        on="date",
        how="inner"
    ).dropna()

    # Need enough observations for regression
    if len(merged) < 30:
        continue

    regression = linregress(
        merged["benchmark_return"],
        merged["daily_return"]
    )

    alpha = regression.intercept * 252
    beta = regression.slope

    alpha_beta_results.append({
        "scheme_code": scheme_code,
        "alpha": alpha,
        "beta": beta,
        "observations": len(merged)
    })

alpha_beta = pd.DataFrame(alpha_beta_results)

alpha_beta.head()

KeyError: 'close'

# Combining Everything

In [ ]:
performance = (
    cagr
    .merge(
        sharpe,
        on="scheme_code",
        how="left"
    )
    .merge(
        sortino,
        on="scheme_code",
        how="left"
    )
    .merge(
        drawdown[
            [
                "scheme_code",
                "max_drawdown",
                "drawdown_date"
            ]
        ],
        on="scheme_code",
        how="left"
    )
    .merge(
        alpha_beta[
            [
                "scheme_code",
                "alpha",
                "beta",
                "r_squared"
            ]
        ],
        on="scheme_code",
        how="left"
    )
)

display(performance)